In [16]:
!pip install xarray netCDF4 emcee corner h5netcdf scikit-learn
import numpy as np
import xarray as xr
import pandas as pd
import emcee
import corner
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score




[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
df_1 = pd.read_csv('params.csv')  # Load your data from a CSV file
df_1.head()
df_1['run_id'] = df_1['lut_path'].str.extract(r'lut_(\d+)\.dat$')[0].astype(int)
ds_nc = xr.open_dataset('combined_output.nc',engine='netcdf4')
df_nc = ds_nc.to_dataframe().reset_index()
df_nc = df_nc.dropna(subset=['PRECT'])
df_unified = pd.merge(
    df_1,
    df_nc,
    on='run_id',
    how='inner'           # only keep subjects that appear in BOTH files
)


In [18]:
# Step 1: compute mean of output variables for each run_id
output_cols = ['TGCLDIWP', 'LWCF', 'SWCF', 'CLDTOT', 'PRECT']
df_outputs = df_unified.groupby('run_id')[output_cols].mean().reset_index()

# Step 2: get one row of parameters per run_id (they dont change within a run)
param_cols = ['run_id', 'ds', 'rho_e', 'bas', 'micro_mg_vtrmi_factor', 'micro_mg_berg_eff_factor']
df_params = df_unified[param_cols].drop_duplicates(subset='run_id')

# Step 3: merge them together on run_id
df_model = pd.merge(
    df_params,
    df_outputs,
    on='run_id')

print(df_model.shape)
print(df_model.head())

(499, 11)
   run_id       ds    rho_e      bas  micro_mg_vtrmi_factor  \
0       0  2.43139  0.88300  1.93092                1.55731   
1       1  2.98594  0.80154  1.51213                1.82759   
2       2  1.69070  0.53283  1.96759                4.26234   
3       3  2.28133  0.71636  1.55453                2.53309   
4       4  2.20747  0.85695  1.43424                4.43978   

   micro_mg_berg_eff_factor  TGCLDIWP       LWCF       SWCF    CLDTOT  \
0                   0.10298  0.114449  39.054104 -64.122383  0.713200   
1                   0.47674  0.055845  10.422736 -39.265537  0.688813   
2                   0.50011  0.062612  26.243057 -46.960510  0.651257   
3                   0.89830  0.066999  27.498352 -46.397758  0.661900   
4                   0.65745  0.075053  30.412548 -51.234451  0.658562   

          PRECT  
0  3.164195e-08  
1  3.174044e-08  
2  3.272332e-08  
3  3.239926e-08  
4  3.213558e-08  


In [23]:
X = df_model[['ds', 'rho_e', 'bas', 'micro_mg_vtrmi_factor', 'micro_mg_berg_eff_factor']].values
y = df_model[['TGCLDIWP', 'LWCF', 'SWCF', 'CLDTOT', 'PRECT']].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (499, 5)
y shape: (499, 5)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# train
emulator = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
emulator.fit(X_train_scaled, y_train)

# evaluate
y_pred = emulator.predict(X_test_scaled)
print("R² per output variable:")
for i, col in enumerate(output_cols):
    print(f"  {col}: {r2_score(y_test[:, i], y_pred[:, i]):.3f}")

R² per output variable:
  TGCLDIWP: 0.513
  LWCF: 0.677
  SWCF: 0.504
  CLDTOT: 0.330
  PRECT: 0.418


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# degree=2 adds squared terms and interaction terms between parameters
# e.g. ds², ds×rho_e, rho_e² etc.
poly_emulator = Pipeline([
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('ridge', RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5))
])

poly_emulator.fit(X_train, y_train)
y_pred_poly = poly_emulator.predict(X_test)

print("R² with polynomial features:")
for i, col in enumerate(output_cols):
    print(f"  {col}: {r2_score(y_test[:, i], y_pred_poly[:, i]):.3f}")

R² with polynomial features:
  TGCLDIWP: 0.619
  LWCF: 0.881
  SWCF: 0.757
  CLDTOT: 0.344
  PRECT: 0.518


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_emulator = RandomForestRegressor(n_estimators=100, random_state=42)
rf_emulator.fit(X_train, y_train)
y_pred_rf = rf_emulator.predict(X_test)

print("R² with Random Forest:")
for i, col in enumerate(output_cols):
    print(f"  {col}: {r2_score(y_test[:, i], y_pred_rf[:, i]):.3f}")

R² with Random Forest:
  TGCLDIWP: 0.616
  LWCF: 0.914
  SWCF: 0.864
  CLDTOT: 0.363
  PRECT: 0.463


In [ ]:
def log_likelihood(theta, emulator, x_obs, y_obs, yerr_obs):
    param_1, param_2, param_3 = theta
    
    # ask the emulator: "if these were the true parameters, what output would you predict?"
    predicted = emulator.predict([[param_1, param_2, param_3]])
    
    # compare prediction to what we actually observed
    sigma2 = yerr_obs**2
    return -0.5 * np.sum((y_obs - predicted)**2 / sigma2 + np.log(sigma2))

def log_prior(theta):
    param_1, param_2, param_3 = theta
    if 0 < param_1 < 2 and 0 < param_2 < 2 and 0 < param_3 < 2:
        return 0.0
    return -np.inf

def log_probability(theta, emulator, x_obs, y_obs, yerr_obs):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, emulator, x_obs, y_obs, yerr_obs)

In [ ]:

# Start scipy's search near a rough guess of parameters
initial_guess = np.array([1.0, 0.0, np.log(0.5)])
# minimize the NEGATIVE log likelihood
x = df_unified[['TGCLDIWP', 'LWCF', 'SWCF', 'CLDTOT', 'PRECT']].values      # .values converts pandas Series to numpy array
y = df_unified[['ds', 'rho_e', 'bas', 'micro_mg_vtrmi_factor', 'micro_mg_berg_eff_factor']].values
yerr = np.ones(y.shape) * 0.1
result = minimize(
    lambda theta, x, y, yerr: -log_likelihood(theta, x, y, yerr),
    initial_guess,
    args=(x, y, yerr)
)

best_params = result.x    # maximum likelihood estimate
print("Best starting params:", best_params)

TypeError: log_likelihood() missing 1 required positional argument: 'yerr_obs'

In [ ]:
nwalkers = 32
ndim = 3    # number of unknowns (m, b, log_f)

# tiny random scatter around the best params
# 1e-4 means a very small nudge so walkers start slightly different from each other
pos = best_params + 1e-4 * np.random.randn(nwalkers, ndim)

In [ ]:
sampler = emcee.EnsembleSampler(
    nwalkers, ndim, log_probability, args=(x, y, yerr)
)

sampler.run_mcmc(pos, 150, progress=True)

100%|██████████| 150/150 [32:33<00:00, 13.02s/it]


State([[ 0.43828517  1.2389242  -0.01113466]
 [ 0.4383378   1.23968023 -0.01164598]
 [ 0.43857613  1.24027051 -0.01148549]
 [ 0.43886199  1.24091778 -0.01225097]
 [ 0.43875355  1.23996306 -0.01199646]
 [ 0.43848077  1.24009474 -0.01175549]
 [ 0.43862879  1.23973686 -0.01173729]
 [ 0.4386665   1.23906809 -0.01120765]
 [ 0.43832705  1.239454   -0.01100964]
 [ 0.43882985  1.2403751  -0.01256006]
 [ 0.43841301  1.24004017 -0.01210378]
 [ 0.43856164  1.23992749 -0.01158341]
 [ 0.43899143  1.24102123 -0.0121722 ]
 [ 0.43869958  1.23961506 -0.01128731]
 [ 0.43876593  1.24020127 -0.01225577]
 [ 0.43876832  1.24093485 -0.01124811]
 [ 0.43824083  1.23969652 -0.01207749]
 [ 0.43828413  1.24004255 -0.01151924]
 [ 0.43868097  1.23991273 -0.01166265]
 [ 0.43889263  1.24000621 -0.01126124]
 [ 0.43834402  1.24005614 -0.01144452]
 [ 0.43846852  1.24079821 -0.01211608]
 [ 0.43860957  1.24046246 -0.01190167]
 [ 0.43849752  1.2401988  -0.01171138]
 [ 0.43876787  1.24033886 -0.01185127]
 [ 0.43874998  1.24

In [ ]:
tau = sampler.get_autocorr_time()
print("Autocorrelation times:", tau)
burnin = int(2 * np.max(tau))     
thin = int(0.5 * np.min(tau))  

NameError: name 'sampler' is not defined

In [ ]:
flat_samples = sampler.get_chain(discard=burnin, thin=thin, flat=True)
print("Sample shape:", flat_samples.shape)

In [ ]:
labels = ["m", "b", "log(f)"]
fig = corner.corner(flat_samples, labels=labels)
plt.show()

In [ ]:
for i in range(ndim):
    mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])
    q = np.diff(mcmc)
    print(f"{labels[i]} = {mcmc[1]:.3f} +{q[1]:.3f} / -{q[0]:.3f}")